In [1]:
%pip install geopandas pyarrow

In [2]:
import urllib.request, zipfile, shutil, tempfile
from pathlib import Path

# --- Vérification du lakehouse ---
LH_FILES = Path('/lakehouse/default/Files')
if not LH_FILES.is_dir():
    raise RuntimeError(
        "Aucun lakehouse par défaut monté. Ajoute lh_mobility_citiz (type Lakehouse) "
        "dans l'explorateur, vérifie l'épingle, puis redémarre la session."
    )
print("Lakehouse monté :", LH_FILES)

bronze_dir = LH_FILES / 'bronze' / 'insee_population'
silver_dir = LH_FILES / 'silver' / 'insee_population'
bronze_dir.mkdir(parents=True, exist_ok=True)
silver_dir.mkdir(parents=True, exist_ok=True)

url_insee = "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_gpkg.zip"
zip_name = "Filosofi2021_carreaux_200m_gpkg.zip"
zip_lakehouse = bronze_dir / zip_name

# --- Travail sur le disque local (plus fiable que le montage OneLake) ---
work_dir = Path(tempfile.mkdtemp())
zip_local = work_dir / zip_name
extract_local = work_dir / "extracted"

print("\n=== COUCHE BRONZE ===")
if zip_lakehouse.exists():
    print("ZIP déjà présent dans le lakehouse, copie locale...")
    shutil.copy(zip_lakehouse, zip_local)
else:
    print("Téléchargement depuis l'INSEE...")
    urllib.request.urlretrieve(url_insee, zip_local)
    print("Sauvegarde du ZIP dans le lakehouse...")
    shutil.copy(zip_local, zip_lakehouse)

print("Extraction...")
with zipfile.ZipFile(zip_local) as z:
    z.extractall(extract_local)

gpkg_files = list(extract_local.rglob("*.gpkg"))
if not gpkg_files:
    raise ValueError("Aucun .gpkg trouvé dans le ZIP.")
print("GeoPackages trouvés :", [f.name for f in gpkg_files])

met = [f for f in gpkg_files if 'met' in f.name.lower()]
gpkg_path = met[0] if met else gpkg_files[0]   # chemin LOCAL

# Archive du .gpkg extrait dans Bronze
(bronze_dir / "extracted").mkdir(exist_ok=True)
shutil.copy(gpkg_path, bronze_dir / "extracted" / gpkg_path.name)

print(f"\nBronze OK -> {bronze_dir}")
print(f"GeoPackage sélectionné : {gpkg_path.name}")

In [4]:
import geopandas as gpd
import pandas as pd
import notebookutils, tempfile
from pathlib import Path

print("=== COUCHE SILVER ===")

# --- 0. Récupération du GeoPackage depuis Bronze ---
rel_gpkg = "Files/bronze/insee_population/extracted/carreaux_200m_met.gpkg"
rel_silver = "Files/silver/insee_population"

if not notebookutils.fs.exists(rel_gpkg):
    raise FileNotFoundError(f"{rel_gpkg} introuvable : relance la cellule Bronze.")

work_dir = Path(tempfile.mkdtemp())
gpkg_path = work_dir / "carreaux_200m_met.gpkg"
print("Copie locale du GeoPackage...")
notebookutils.fs.cp(rel_gpkg, f"file:{gpkg_path}")

# --- 1. Identification des colonnes ---
apercu = gpd.read_file(gpkg_path, rows=1)
colonnes = [c for c in apercu.columns if c != 'geometry']
print("Colonnes disponibles :", colonnes)

candidats_id = [c for c in colonnes if c.lower() in ('idcar_200m', 'idinspire')] \
            or [c for c in colonnes if 'id' in c.lower()]
candidats_pop = [c for c in colonnes if c.lower() in ('ind', 'ind_snv', 'population')]
if not candidats_id or not candidats_pop:
    raise ValueError(f"Colonnes ID/population introuvables. Disponibles : {colonnes}")
col_id, col_pop = candidats_id[0], candidats_pop[0]
print(f"ID -> '{col_id}' | Population -> '{col_pop}'")

# --- 2. Lecture des colonnes utiles ---
print("Chargement du GeoPackage (1 à 3 min)...")
gdf = gpd.read_file(gpkg_path, columns=[col_id, col_pop])
print(f"{len(gdf):,} carreaux chargés | CRS : {gdf.crs}")

# --- 3. Centroïdes en Lambert 93 puis WGS84 ---
centroides = gdf.geometry.centroid.to_crs(epsg=4326)

df_silver = pd.DataFrame({
    'IdINSPIRE': gdf[col_id].values,
    'Ind': gdf[col_pop].values,
    'longitude': centroides.x.values,
    'latitude': centroides.y.values,
})
print(f"\nNombre total de carreaux : {len(df_silver):,}")
display(df_silver.head())

# --- 4. Écriture locale puis copie dans le lakehouse (écrase si existe) ---
local_parquet = work_dir / "carreaux_population_gps.parquet"
df_silver.to_parquet(local_parquet, index=False)

dest_parquet = f"{rel_silver}/carreaux_population_gps.parquet"
notebookutils.fs.mkdirs(rel_silver)
if notebookutils.fs.exists(dest_parquet):
    notebookutils.fs.rm(dest_parquet)
notebookutils.fs.cp(f"file:{local_parquet}", dest_parquet)
print(f"\nSilver OK -> {dest_parquet}")

In [4]:
import geopandas as gpd
import pandas as pd
import numpy as np
import notebookutils, tempfile
from pathlib import Path

print("=== SILVER : CARREAUX POPULATION + REVENUS ===")

rel_extracted = "Files/bronze/insee_population/filosofi2021/extracted"
rel_silver    = "Files/silver/insee_population"
territoires = {"met": "Métropole", "mart": "Martinique", "reun": "La Réunion"}

def chercher_gpkg(dossier):
    """Parcourt récursivement un dossier OneLake et renvoie les .gpkg trouvés."""
    trouves = []
    for f in notebookutils.fs.ls(dossier):
        if f.isDir:
            trouves += chercher_gpkg(f.path)
        elif f.name.lower().endswith(".gpkg"):
            trouves.append(f)
    return trouves

gpkgs = chercher_gpkg("Files/bronze/insee_population")
for f in gpkgs:
    print("Trouvé :", f.path)
if not gpkgs:
    raise FileNotFoundError("Aucun .gpkg dans Files/bronze/insee_population : relance la cellule Bronze.")

work_dir = Path(tempfile.mkdtemp())
morceaux = []
# https://www.insee.fr/fr/statistiques/8735162?sommaire=8735243
for f in gpkgs:
    code = f.name.replace("carreaux_200m_", "").replace(".gpkg", "")
    local = work_dir / f.name
    notebookutils.fs.cp(f.path, f"file:{local}")

    print(f"\nLecture {f.name}...")
    gdf = gpd.read_file(local)
    gdf.columns = [c.lower() if c != "geometry" else c for c in gdf.columns]
    print(f"  {len(gdf):,} carreaux | CRS : {gdf.crs}")

    # Centroïde dans le CRS natif (Lambert 93 / UTM DROM) puis WGS84
    centro = gdf.geometry.centroid.to_crs(epsg=4326)
    df = pd.DataFrame(gdf.drop(columns="geometry"))
    df["longitude"] = centro.x.values
    df["latitude"] = centro.y.values
    df["territoire"] = territoires.get(code, code)
    morceaux.append(df)

df = pd.concat(morceaux, ignore_index=True)

# --- Indicateurs dérivés (division protégée contre les zéros) ---
def ratio(num, den):
    return np.where(df[den] > 0, df[num] / df[den], np.nan)

df["niveau_vie_moyen"]   = ratio("ind_snv", "ind").round(0)
df["taux_pauvrete_men"]  = ratio("men_pauv", "men").round(3)
df["part_log_sociaux"]   = ratio("log_soc", "men").round(3)
df["part_proprietaires"] = ratio("men_prop", "men").round(3)
df["part_65_plus"]       = np.where(df["ind"] > 0, (df["ind_65_79"] + df["ind_80p"]) / df["ind"], np.nan).round(3)
df["part_18_24"]         = ratio("ind_18_24", "ind").round(3)
df["code_commune"]       = df["lcog_geo"].astype(str).str[:5]   # commune principale
df["est_impute"]         = df["i_est_200"].astype(int) == 1

df = df.rename(columns={"idcar_200m": "IdINSPIRE", "ind": "Ind"})

print(f"\nTotal : {len(df):,} carreaux")
display(df[["IdINSPIRE", "territoire", "Ind", "niveau_vie_moyen",
            "taux_pauvrete_men", "part_65_plus", "latitude", "longitude"]].head())

# --- Écriture locale puis copie dans le lakehouse ---
local_parquet = work_dir / "carreaux_revenus_gps.parquet"
df.to_parquet(local_parquet, index=False)

dest = f"{rel_silver}/carreaux_revenus_gps.parquet"
notebookutils.fs.mkdirs(rel_silver)
if notebookutils.fs.exists(dest):
    notebookutils.fs.rm(dest)
notebookutils.fs.cp(f"file:{local_parquet}", dest)
print(f"\nSilver OK -> {dest}")